# Object detection with audio feedback

Upload a photo, find the objects in it with a pretrained YOLO model, draw boxes with labels, and read out what was found.


In [ ]:
!pip install -q ultralytics gTTS

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
from google.colab import files
from gtts import gTTS
from IPython.display import Audio, display
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

model = YOLO("yolo11n.pt")  # small pretrained model, no training here

Describe the result as counts ("2 persons, 1 sports ball") instead of repeating a name for every box.

In [ ]:
def describe(names):
    if not names:
        return "No objects detected."
    parts = []
    for name, n in Counter(names).items():
        parts.append(f"{n} {name}" if n == 1 else f"{n} {name}s")
    return "Detected " + ", ".join(parts) + "."

In [ ]:
def draw_boxes(image, boxes, names):
    draw = ImageDraw.Draw(image)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", size=30)
    except OSError:
        font = ImageFont.load_default()

    for box, name in zip(boxes, names):
        x1, y1, x2, y2 = box
        draw.rectangle([x1, y1, x2, y2], outline="blue", width=3)

        # label sits on a filled background so it stays readable
        left, top, right, bottom = draw.textbbox((x1, y1), name, font=font)
        h = bottom - top
        draw.rectangle([x1, y1 - h - 6, x1 + (right - left) + 6, y1], fill="blue")
        draw.text((x1 + 3, y1 - h - 4), name, fill="white", font=font)
    return image

Upload an image and run the detector.

In [ ]:
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

results = model(image)
boxes = results[0].boxes.xyxy.cpu().numpy()
names = [model.names[int(c)] for c in results[0].boxes.cls.cpu().numpy()]

plt.figure(figsize=(10, 10))
plt.imshow(draw_boxes(image, boxes, names))
plt.axis("off")
plt.show()

description = describe(names)
print(description)

gTTS(description).save("detection_audio.mp3")
display(Audio("detection_audio.mp3", autoplay=True))